In [1]:
import os
import json

from dotenv import load_dotenv
from google import genai
from google.genai import types

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

In [2]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    *   Visit [https://ollama.com/download](https://ollama.com/download).
    *   Choose your operating system:
        *   **macOS**: Download and install the `.pkg` file.
        *   **Windows**: Download and install the `.msi` file.
        *   **Linux**: Run the following command in your terminal:
            ```bash
            curl -fsSL https://ollama.com/install.sh | sh
            ```

2.  **Run a Model (e.g., LLaMA 3):**
    *   Open a terminal and type:
        ```bash
        ollama run llama3
        ```
    *   This command will download the LLaMA 3 model (~4GB), start it locally, and open a chat-like interface.

3.  **Test the Ollama Local Server:**
    *   Run the following command in your terminal:
        ```bash
        curl http://localhost:11434
        ```
    *   You should receive a JSON response similar to `{"models": [...]}`.

4.  **Install the Python Client (Optional, for programmatic interac

In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

Based on the provided context, there is no information on how to run Olama locally. The context mentions that the course can be run locally if you are comfortable setting up tools like Python, `uv`, Jupyter, and Docker, but it does not provide specific instructions for Olama.


In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
# 2. Convert OpenAI schema to Gemini's expected Pydantic format
# Note: Root "type": "function" and "additionalProperties" are removed.
# Types are capitalized ("OBJECT", "STRING").
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='search',
            description='Search the FAQ database for entries matching the given query.',
            parameters=types.Schema(
                type="OBJECT",
                properties={
                    'query': types.Schema(
                        type="STRING",
                        description='Search query text to look up in the course FAQ.'
                    )
                },
                required=["query"]
            )
        )
    ]
)

# 3. Format the conversation history for Gemini
messages = [
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text="I just discovered the course. Can I join it?")]
    )
]

# 4. Set up the config configuration wrapper
config = types.GenerateContentConfig(
    tools=[search_tool]
)

# 5. Make the API call (replaces openai_client.responses.create)
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=messages,
    config=config
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 10.212044497s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [ ]:
# 6. Extract outputs (replaces response.output_text)
if response.function_calls:
    # The model decided it needs to search the FAQ database
    for call in response.function_calls:
        print(f"Tool called: {call.name}")
        print(f"Arguments: {call.args}") # e.g., {'query': 'grading policy'}
else:
    # The model answered directly without needing the tool
    print(response.text)

In [ ]:
response.function_calls[0].args

In [ ]:
args = response.function_calls[0].args
args

In [ ]:
results = search(**args)

In [ ]:
result_json = json.dumps(results, indent=2)
print(result_json)

In [ ]:
len(results)

In [ ]:
# 1. Append the model's tool request (Replaces messages.extend(response.text))
# Gemini requires storing the function call structure back into history
messages.append(
    types.Content(
        role="assistant",
        parts=[
            types.Part.from_function_call(
                name=call.name,
                args=call.args
            )
        ]
    )
)

# 2. Append your FAQ database search result (Replaces your dictionary append)
# The result parameter must be a dictionary, which perfectly accepts your result_json
messages.append(
    types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(
                name=call.name,
                response={"result": result_json} 
            )
        ]
    )
)

In [ ]:
# Send the history containing the tool output back to Gemini
final_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=messages,
    config=config
)

print(final_response.text)

### Agentic Loop

In [ ]:
def make_call(call):
    # 1. No json.loads() needed! call.args is already a Python dictionary.
    args = call.args

    if call.name == 'search':
        # Safely unpack arguments into your search function
        result = search(**args)
    else:
        result = {"error": f"Unknown tool: {call.name}"}

    # 2. Return a Gemini Content object with the 'tool' role.
    # Note: No json.dumps() needed; response accepts a dictionary.
    return types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(
                name=call.name,
                response={"result": result}
            )
        ]
    )

In [ ]:
# 2. Define the System Instructions (Passed via config, not messages)
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

# 3. Define the tool schema using Gemini's native types.Tool format
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='search',
            description='Search the FAQ database for entries matching the given query.',
            parameters=types.Schema(
                type="OBJECT",
                properties={
                    'query': types.Schema(
                        type="STRING",
                        description='Search query text to look up in the course FAQ.'
                    )
                },
                required=["query"]
            )
        )
    ]
)

# 4. Construct the GenerateContentConfig object
config = types.GenerateContentConfig(
    system_instruction=instructions,  # <-- Core Mapping Change
    tools=[search_tool]
)

# 5. Format the Conversation History for Gemini (User and Assistant turns only)
question = 'I just discovered the course. Can I join it?'

messages = [
    types.Content(
        role='user',
        parts=[types.Part.from_text(text=question)]
    )
]

# 6. Execute the initial API Call
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=messages,
    config=config
)

In [ ]:
response

In [ ]:
# 1. Append the model's response back to history if Gemini requested a function call
if response.function_calls:
    for call in response.function_calls:
        # Save the assistant's tool call turn to history (Replaces messages.extend)
        messages.append(
            types.Content(
                role="assistant",
                parts=[
                    types.Part.from_function_call(
                        name=call.name,
                        args=call.args
                    )
                ]
            )
        )
        
        # 2. Replaces item.type == 'function_call' block
        print('function_call:', call.name, call.args)
        
        # Execute your converted make_call() function and append output
        call_output = make_call(call)
        messages.append(call_output)

# 3. Replaces item.type == 'message' block
elif response.text:
    print('ASSISTANT:')
    print(response.text)

In [ ]:
messages

In [ ]:
len(messages)

In [ ]:
# 3. Initialize Conversation History (User turns only)
question = 'I just discovered the course. Can I join it?'
messages = [
    types.Content(
        role='user',
        parts=[types.Part.from_text(text=question)]
    )
]

it = 1

# 4. Core Execution Loop
while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=messages,
        config=config
    )

    # Process function calls if the model requests them
    if response.function_calls:
        has_function_calls = True
        for call in response.function_calls:
            # Step A: Append assistant's function call intent to history
            messages.append(
                types.Content(
                    role="assistant",
                    parts=[
                        types.Part.from_function_call(
                            name=call.name,
                            args=call.args
                        )
                    ]
                )
            )
            
            print('function_call:', call.name, call.args)
            
            # Step B: Run the function and append tool results to history
            call_output = make_call(call)
            messages.append(call_output)

    # Process text response if the model outputs final text
    if response.text:
        print('ASSISTANT:')
        print(response.text)
    
    it = it + 1
    if not has_function_calls:
        break